In [14]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
import torch.optim as optim

from smallcnn import SmallImageCNN
from pathlib import Path

In [15]:


train_transform = transforms.Compose([
    transforms.Resize((64, 64)),  
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
test_transforms = transforms.Compose([
    transforms.Resize((64, 64)),         # Ensure all inputs are exactly 64x64
    transforms.RandomHorizontalFlip(),   # Simple augmentation to prevent overfitting
    transforms.ToTensor(),               # Convert to pixel values 0-1
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # Standard normalization
])
# Load images from folders
train_dataset = datasets.ImageFolder(root='VisDrone-objects/train', transform=train_transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

val_dataset = datasets.ImageFolder(root='VisDrone-objects/val', transform=test_transforms)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [16]:

# 1. Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SmallImageCNN(num_classes=10).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()


In [17]:
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=5
)

In [18]:
def validate_model(model, val_loader, criterion, device):
    model.eval()  # Set to evaluation mode
    val_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():  # Turn off gradients to save memory and speed up
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            
            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * images.size(0)
            
            # Calculate accuracy
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
    avg_loss = val_loss / len(val_loader.dataset)
    accuracy = 100 * correct / total
    return avg_loss, accuracy

# Usage inside your training loop:
# val_loss, val_acc = validate_model(model, val_loader, criterion, device)
# print(f"Validation Accuracy: {val_acc:.2f}%")

In [19]:

log_dir = Path('smallcnn_logs/2')
log_dir.mkdir(exist_ok=True)

In [20]:


# Configuration
num_epochs = 50
best_val_acc = 0.0

for epoch in range(num_epochs):
    # --- TRAINING PHASE ---
    model.train()
    running_loss = 0.0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()

    # --- VALIDATION PHASE ---
    # Using the validate_model function from the previous step
    val_loss, val_acc = validate_model(model, val_loader, criterion, device)
    
    scheduler.step(val_acc)
    
    print(f"Epoch [{epoch+1}/{num_epochs}] | Train Loss: {running_loss/len(train_loader):.4f} | Val Acc: {val_acc:.2f}%")
    
    # --- SAVING PROGRESS ---
    # 1. Save the "Latest" version (overwrites every epoch)
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': loss.item(),
    }, f'{log_dir}/last_checkpoint.pth')
    
    # 2. Save the "Best" version (only if accuracy improved)
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), f'{log_dir}/best_model.pth')
        print(f"--> New best model saved with {val_acc:.2f}% accuracy!")


Epoch [1/50] | Train Loss: 1.2551 | Val Acc: 61.41%
--> New best model saved with 61.41% accuracy!
Epoch [2/50] | Train Loss: 1.0847 | Val Acc: 63.69%
--> New best model saved with 63.69% accuracy!
Epoch [3/50] | Train Loss: 1.0208 | Val Acc: 64.75%
--> New best model saved with 64.75% accuracy!
Epoch [4/50] | Train Loss: 0.9809 | Val Acc: 66.81%
--> New best model saved with 66.81% accuracy!
Epoch [5/50] | Train Loss: 0.9540 | Val Acc: 66.01%
Epoch [6/50] | Train Loss: 0.9337 | Val Acc: 68.09%
--> New best model saved with 68.09% accuracy!
Epoch [7/50] | Train Loss: 0.9180 | Val Acc: 68.54%
--> New best model saved with 68.54% accuracy!
Epoch [8/50] | Train Loss: 0.9041 | Val Acc: 68.88%
--> New best model saved with 68.88% accuracy!
Epoch [9/50] | Train Loss: 0.8927 | Val Acc: 69.07%
--> New best model saved with 69.07% accuracy!
Epoch [10/50] | Train Loss: 0.8831 | Val Acc: 68.08%
Epoch [11/50] | Train Loss: 0.8752 | Val Acc: 68.54%
Epoch [12/50] | Train Loss: 0.8652 | Val Acc: 68.4